# <font color="#418FDE" size="6.5" uppercase>**Pandas & Plots**</font>

>Last update: 20260314.
    
By the end of this Lecture, you will be able to:
- Manipulate civil engineering tables using pandas operations such as filtering, grouping, and joining. 
- Handle missing values and basic data type conversions in preparation for later modeling. 
- Create clear static plots that communicate key trends in civil engineering datasets. 


## **1. Pandas Table Operations**

### **1.1. Reading CE Tables**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_01_01.jpg?v=1773537571" width="250">



>* Import tables preserving meaning, units, and context.
>* Ensure correct names, types; avoid text numbers.

>* Fix messy headers, notes, and split blocks
>* Check formatting, then verify imported columns

>* Import stable IDs as strings for joins
>* Standardize categories, time, and location fields



In [3]:
#@title Python Code - Reading CE Tables

# Read civil engineering tables with reliable column types.
# Handle messy headers and missing values safely.
# Preview key fields before later joins and groups.

# !pip -q install pandas matplotlib

import pandas as pd
import numpy as np

# url = "https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv"
# path = "./datasets/bim.csv"

# Download a small civil engineering CSV dataset.
!wget -q -O bim.csv https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv

# Read identifiers as strings to preserve leading zeros.
df_raw = pd.read_csv('bim.csv')

# Standardize column names for predictable later operations.
df_raw.columns = df_raw.columns.str.strip().str.replace(" ", "_")

# Convert date columns, coercing bad strings into missing values.
df_raw["Start_Date"] = pd.to_datetime(df_raw["Start_Date"], errors="coerce")
df_raw["End_Date"] = pd.to_datetime(df_raw["End_Date"], errors="coerce")

# Convert numeric columns, coercing stray symbols into missing values.
df_raw["Planned_Cost"] = pd.to_numeric(df_raw["Planned_Cost"], errors="coerce")
df_raw["Actual_Cost"] = pd.to_numeric(df_raw["Actual_Cost"], errors="coerce")

# Count missing values in a few critical civil engineering fields.
missing_counts = df_raw[["Project_ID", "Start_Date", "Planned_Cost"]].isna().sum()

# Create a cleaned copy with simple, explainable missing handling.
df = df_raw.dropna(subset=["Project_ID", "Project_Type"]).copy()

# Fill missing costs with median, keeping units and scale.
median_plan = df["Planned_Cost"].median()
df["Planned_Cost"] = df["Planned_Cost"].fillna(median_plan)

# Show a compact import check without printing full tables.
print("Rows, columns:", df.shape)
print("Missing key fields:")
print(missing_counts.to_string())

# Preview a few cleaned columns to verify types.
preview = df[["Project_ID", "Project_Type", "Start_Date", "Planned_Cost"]].head(3)
print("Preview rows:")
print(preview.to_string(index=False))



Rows, columns: (1000, 28)
Missing key fields:
Project_ID      0
Start_Date      0
Planned_Cost    0
Preview rows:
Project_ID Project_Type Start_Date  Planned_Cost
     PJT_1       Tunnel 2020-01-01      12260784
     PJT_2          Dam 2020-01-02       2369277
     PJT_3     Building 2020-01-03      23299783


### **1.2. Filtering and sorting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_01_02.jpg?v=1773537599" width="250">



>* Filter rows to match engineering conditions.
>* Validate columns, reduce noise, answer questions.

>* Sorting orders filtered data to reveal priorities.
>* Use multi-key, direction, stable sorts.

>* Account for missing, inconsistent, mixed-type data
>* Standardize units, boundaries for reproducible subsets



In [ ]:
#@title Python Code - Filtering and sorting

# Learn pandas filtering for civil engineering tables.
# Practice sorting to prioritize critical asset records.
# Keep outputs short and focused for beginners.

# !pip -q install pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt

# Download a small civil engineering style dataset.
!wget -q -O bim.csv "https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv"

# Load the dataset into a pandas DataFrame.
df = pd.read_csv("bim.csv")

# Show a compact preview of key columns.
cols = ["Project_ID", "Project_Type", "Location", "Cost_Overrun", "Risk_Level"]

# Keep only columns that actually exist.
cols = [c for c in cols if c in df.columns]

# Print a tiny sample, not the full table.
print(df[cols].head(3).to_string(index=False))

# Convert cost overrun to numeric safely.
df["Cost_Overrun"] = pd.to_numeric(df["Cost_Overrun"], errors="coerce")

# Count missing values after conversion.
missing_overrun = int(df["Cost_Overrun"].isna().sum())

# Print one short line about missing values.
print("Missing Cost_Overrun values:", missing_overrun)

# Fill missing overruns with the median value.
median_overrun = float(df["Cost_Overrun"].median(skipna=True))

# Replace missing values using the computed median.
df["Cost_Overrun"] = df["Cost_Overrun"].fillna(median_overrun)

# Filter rows using multiple engineering-style conditions.
mask = (df["Project_Type"].isin(["Bridge", "Road"])) & (df["Cost_Overrun"] > 0)

# Create a focused subset for review.
subset = df.loc[mask, ["Project_ID", "Project_Type", "Cost_Overrun", "Risk_Level"]]

# Sort to prioritize the largest overruns first.
subset_sorted = subset.sort_values(by=["Cost_Overrun", "Project_ID"], ascending=[False, True])

# Print only the top five prioritized records.
print(subset_sorted.head(5).to_string(index=False))

# Group the filtered subset by project type.
grouped = subset_sorted.groupby("Project_Type")["Cost_Overrun"].mean()

# Sort group results to compare typical overruns.
grouped = grouped.sort_values(ascending=False)

# Print a short summary table for decision making.
print(grouped.round(2).to_string())

# Create one clear plot from the grouped results.
ax = grouped.plot(kind="bar", color=["steelblue", "darkorange"])

# Add a readable title and axis labels.
ax.set_title("Average Cost Overrun after Filtering")

# Label axes for clear communication.
ax.set_xlabel("Project Type")

# Label the metric with units context.
ax.set_ylabel("Mean Cost Overrun")

# Keep the plot tidy for slides.
plt.tight_layout()

plt.show()



### **1.3. Group and Aggregate**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_01_03.jpg?v=1773537629" width="250">



>* Group rows by meaningful engineering categories.
>* Aggregate groups into totals, averages, trends.

>* Choose grouping and summaries to match meaning
>* Aggregate multiple columns; use clear names

>* Handle missingness, zeros, and messy group IDs
>* Choose aggregation detail and downstream use



In [ ]:
#@title Python Code - Group and Aggregate

# Learn pandas grouping for civil engineering summaries.
# Practice aggregations with missing values handled safely.
# Create one simple plot from grouped results.

# !pip install pandas matplotlib

# Import core libraries for tables and plots.
import pandas as pd
import matplotlib.pyplot as plt

# Download a small civil engineering style dataset.
!wget -q -O bim.csv "https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv"

# Load the CSV into a pandas DataFrame.
df = pd.read_csv("bim.csv")

# Keep only columns needed for grouping and aggregation.
keep_cols = ["Project_Type", "Planned_Cost", "Actual_Cost", "Labor_Hours"]
df = df[keep_cols]

# Convert numeric columns, forcing bad values to missing.
df["Planned_Cost"] = pd.to_numeric(df["Planned_Cost"], errors="coerce")
df["Actual_Cost"] = pd.to_numeric(df["Actual_Cost"], errors="coerce")

# Convert labor hours, then fill missing with median.
df["Labor_Hours"] = pd.to_numeric(df["Labor_Hours"], errors="coerce")
df["Labor_Hours"] = df["Labor_Hours"].fillna(df["Labor_Hours"].median())

# Create a derived column for cost overrun.
df["Cost_Overrun"] = df["Actual_Cost"] - df["Planned_Cost"]

# Filter out rows missing key cost values.
df_clean = df.dropna(subset=["Planned_Cost", "Actual_Cost"])

# Group by project type and compute multiple summaries.
grouped = df_clean.groupby("Project_Type").agg(
    Mean_Overrun=("Cost_Overrun", "mean"),
    Total_Labor_Hours=("Labor_Hours", "sum"),
    Test_Count=("Cost_Overrun", "count"),
)

# Sort groups by mean overrun for easier reading.
grouped = grouped.sort_values("Mean_Overrun", ascending=False)

# Print a compact summary table with few rows.
print("Grouped summary by Project_Type:")
print(grouped.round(2).head(8))

# Attach group mean overrun back to each row.
df_joined = df_clean.merge(
    grouped[["Mean_Overrun"]],
    left_on="Project_Type",
    right_index=True,
    how="left",
)

# Compute each project overrun relative to its group.
df_joined["Overrun_vs_Group"] = (
    df_joined["Cost_Overrun"] - df_joined["Mean_Overrun"]
)

# Print a tiny sample showing joined group statistics.
cols_show = ["Project_Type", "Cost_Overrun", "Mean_Overrun", "Overrun_vs_Group"]
print("\nExample rows after joining group mean:")
print(df_joined[cols_show].head(5).round(2))

# Plot mean overrun by project type as bars.
ax = grouped["Mean_Overrun"].plot(kind="bar", figsize=(7, 3))

# Add labels and a reference line at zero.
ax.set_title("Mean Cost Overrun by Project Type")
ax.set_ylabel("Mean Overrun (Actual - Planned)")

# Draw a zero line to separate under and over budget.
ax.axhline(0, color="black", linewidth=1)
plt.tight_layout()

plt.show()



## **2. Cleaning and Type Casting**

### **2.1. Finding Missing Values**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_02_01.jpg?v=1773537659" width="250">



>* Missing data can skew analysis and models.
>* Spot blanks, placeholders, and sentinel values.

>* Scan overall missingness, then check key columns
>* Look for patterns by location, time, events

>* Check types; formatting can hide missing values.
>* Use sanity checks to flag invalid entries.



In [ ]:
#@title Python Code - Finding Missing Values

# Learn to spot missing values in civil datasets.
# Use pandas checks for blanks and sentinels.
# Summarize missingness before cleaning and modeling.

# Import pandas for table operations.
import pandas as pd
# Import numpy for NaN handling.
import numpy as np

# Download a small civil engineering dataset.
!wget -q -O bim.csv "https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv"
# Load the dataset into a DataFrame.
df = pd.read_csv("bim.csv")

# Show dataset size and column count.
print("Rows, Columns:", df.shape)
# Preview a few column names only.
print("First columns:", list(df.columns[:6]))

# Count standard missing values per column.
missing_counts = df.isna().sum()
# Sort missing counts to see worst columns.
missing_sorted = missing_counts.sort_values(ascending=False)

# Print top columns with missing values.
print("Top missing columns:")
print(missing_sorted.head(6))

# Define common placeholder strings for missingness.
placeholders = ["-", "unknown", "Unknown", "N/A", "na"]
# Count placeholder occurrences across the whole table.
placeholder_total = df.isin(placeholders).sum().sum()

# Print placeholder count for quick awareness.
print("Placeholder cells found:", int(placeholder_total))
# Replace placeholders with real NaN values.
df = df.replace(placeholders, np.nan)

# Pick a numeric column often used in modeling.
num_col = "Temperature" if "Temperature" in df.columns else df.columns[0]
# Convert numeric-like text into real numbers.
df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

# Count new missing values after numeric conversion.
new_missing = int(df[num_col].isna().sum())
# Print conversion result for that column.
print("Missing after casting", num_col + ":", new_missing)

# Treat sentinel values as missing for that column.
df.loc[df[num_col].isin([-1, 9999]), num_col] = np.nan
# Print final missing count after sentinel cleanup.
print("Missing after sentinel cleanup:", int(df[num_col].isna().sum()))

# Show a small sample of problematic rows.
bad_rows = df[df[num_col].isna()].head(3)
# Print only two columns to stay compact.
print(bad_rows[[num_col]].to_string(index=False))



### **2.2. Impute or Drop**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_02_02.jpg?v=1773537684" width="250">



>* Choose impute or drop based on meaning.
>* Dropping loses cases; imputation adds assumptions.

>* Drop only when missingness is minor, irrelevant.
>* Systematic dropping biases results; check remaining representativeness.

>* Impute to keep records, using context-aware methods.
>* Flag imputed values; avoid biasing extremes.



In [ ]:
#@title Python Code - Impute or Drop

# Learn when to impute or drop missing values.
# Practice simple type casting for later modeling.
# Create a small plot showing cleaning impacts.

# !pip -q install pandas matplotlib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
np.random.seed(7)

# Create a tiny bridge inspection style dataset.
df = pd.DataFrame({
    "Bridge_ID": ["B01", "B02", "B03", "B04", "B05", "B06"],
    "Year": [2020, 2020, 2021, 2021, 2022, 2022],
    "Crack_Width_mm": [0.20, np.nan, 0.35, 0.10, np.nan, 0.50],
    "Vibration_Hz": [12.1, 11.8, np.nan, 12.5, 12.0, np.nan],
    "Traffic_Class": ["High", "Low", None, "Medium", "High", "Low"],
})

# Show missing counts before any cleaning.
missing_before = df.isna().sum()
print("Missing values before cleaning:")
print(missing_before.to_string())

# Convert Year to integer safely.
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Year"] = df["Year"].astype("Int64")

# Add flags so models can learn missingness patterns.
df["Crack_missing"] = df["Crack_Width_mm"].isna()
df["Vib_missing"] = df["Vibration_Hz"].isna()

# Impute crack width using median within each year.
crack_median = df.groupby("Year")["Crack_Width_mm"].transform("median")
df["Crack_Width_mm"] = df["Crack_Width_mm"].fillna(crack_median)

# Impute vibration using overall median as fallback.
vib_median = df["Vibration_Hz"].median()
df["Vibration_Hz"] = df["Vibration_Hz"].fillna(vib_median)

# Drop rows missing a key categorical field.
df_drop = df.dropna(subset=["Traffic_Class"]).copy()
df_drop["Traffic_Class"] = df_drop["Traffic_Class"].astype("category")

# Summarize how many rows remain after dropping.
print("\nRows before drop:", len(df), "Rows after drop:", len(df_drop))

# Compare yearly mean crack width after imputation.
yearly_mean = df_drop.groupby("Year")["Crack_Width_mm"].mean()
print("\nYearly mean crack width after imputation:")
print(yearly_mean.round(3).to_string())

# Plot the trend to communicate the cleaned result.
plt.figure(figsize=(6, 3))
plt.plot(yearly_mean.index.astype(int), yearly_mean.values, marker="o")
plt.title("Mean Crack Width by Year (Impute + Drop)")
plt.xlabel("Inspection Year")
plt.ylabel("Mean Crack Width (mm)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



### **2.3. Type Casting Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_02_03.jpg?v=1773537713" width="250">



>* Set each column to correct data type.
>* Wrong types break calculations; fix before modeling.

>* Cast to numeric, datetime, or categories.
>* Reveals issues; aligns data with meaning.

>* Choose numeric types carefully; keep IDs as text.
>* Watch units and missing values during casting.



In [ ]:
#@title Python Code - Type Casting Basics

# Learn basic type casting for civil engineering tables.
# Convert text columns into numeric, datetime, and category.
# Handle messy entries and missing values safely.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create a small messy inspection style dataset.
df = pd.DataFrame({
    "Asset_ID": ["0012", "0013", "0014", "0015", "0016"],
    "Start_Date": ["2024-01-05", "2024/01/06", "bad_date", None, "2024-01-09"],
    "PCI": ["85", ">90", "N/A", "72 approx", None],
    "Lanes": ["2", "3", None, "4", "two"],
    "Surface": ["Asphalt", "asphalt", "CONCRETE", "Asphalt ", None],
})

# Show initial dtypes to reveal text storage.
print("Initial dtypes:")
print(df.dtypes)

# Keep identifiers as text to preserve leading zeros.
df["Asset_ID"] = df["Asset_ID"].astype("string")

# Convert dates using coercion for invalid entries.
df["Start_Date"] = pd.to_datetime(
    df["Start_Date"],
    errors="coerce",
)

# Clean numeric text, then cast to numeric safely.
df["PCI"] = df["PCI"].astype("string")
df["PCI"] = df["PCI"].str.replace(">", "", regex=False)

# Extract digits and decimals, then convert to numbers.
df["PCI"] = df["PCI"].str.extract(
    r"(\d+(?:\.\d+)?)",
    expand=False,
)

# Convert to numeric, turning failures into missing values.
df["PCI"] = pd.to_numeric(
    df["PCI"],
    errors="coerce",
)

# Convert lanes to integer with missing support.
df["Lanes"] = pd.to_numeric(
    df["Lanes"],
    errors="coerce",
)

# Use pandas nullable integer for missing lane counts.
df["Lanes"] = df["Lanes"].astype("Int64")

# Standardize labels, then cast to category.
df["Surface"] = df["Surface"].astype("string")
df["Surface"] = df["Surface"].str.strip()

# Normalize case to reduce accidental duplicate groups.
df["Surface"] = df["Surface"].str.title()
df["Surface"] = df["Surface"].astype("category")

# Report missing values created by safe casting.
missing_counts = df.isna().sum()
print("\nMissing values after casting:")
print(missing_counts)

# Show final dtypes to confirm intended meanings.
print("\nFinal dtypes:")
print(df.dtypes)

# Compute a simple summary without printing full dataframe.
mean_pci = float(df["PCI"].mean(skipna=True))
valid_dates = int(df["Start_Date"].notna().sum())

# Print compact quality checks for modeling readiness.
print("\nQuick checks:")
print("Mean PCI (numeric):", round(mean_pci, 2))
print("Valid Start_Date rows:", valid_dates)

# Plot PCI values to show numeric behavior after casting.
plt.figure(figsize=(6, 3))
plt.plot(df["Asset_ID"], df["PCI"], marker="o")

# Add labels and a clear title for interpretation.
plt.xlabel("Asset_ID")
plt.ylabel("PCI")
plt.title("Pavement Condition Index after Type Casting")

# Show the plot as the single visual output.
plt.tight_layout()
plt.show()



## **3. Static Plot Essentials**

### **3.1. Line and bar plots**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_03_01.jpg?v=1773537749" width="250">



>* Use line plots for ordered sequences.
>* Use bar plots to compare categories.

>* Lines suggest continuity; bars suit discrete groups.
>* Aggregate and group data to match decisions.

>* Use intuitive axes, scales, and bar ordering.
>* Limit lines; label clearly with purposeful colors.



In [ ]:
#@title Python Code - Line and bar plots

# Learn line plots for ordered civil engineering sequences.
# Learn bar plots for comparing grouped engineering categories.
# Use pandas summaries to create clear static plots.

# Import core libraries for tables and plotting.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download a small civil engineering style dataset.
!wget -q -O bim.csv "https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv"

# Load the dataset into a pandas DataFrame.
df = pd.read_csv("bim.csv")

# Keep only columns needed for this plotting lesson.
cols = ["Start_Date", "Project_Type", "Actual_Cost"]
df = df.loc[:, cols]

# Convert dates and costs into usable numeric types.
df["Start_Date"] = pd.to_datetime(df["Start_Date"], errors="coerce")
df["Actual_Cost"] = pd.to_numeric(df["Actual_Cost"], errors="coerce")

# Drop rows missing key values for honest plots.
df = df.dropna(subset=["Start_Date", "Actual_Cost", "Project_Type"])

# Create a monthly time series using grouped averages.
df["Month"] = df["Start_Date"].dt.to_period("M").dt.to_timestamp()
monthly = df.groupby("Month", as_index=False)["Actual_Cost"].mean()

# Create a category summary for a bar plot.
by_type = df.groupby("Project_Type", as_index=False)["Actual_Cost"].mean()
by_type = by_type.sort_values("Actual_Cost", ascending=False)

# Print a tiny preview to confirm transformations worked.
print("Rows after cleaning:", len(df))
print("Months plotted:", len(monthly), "Types plotted:", len(by_type))

# Build one figure with a line plot and bar plot.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot ordered monthly averages as a line plot.
axes[0].plot(monthly["Month"], monthly["Actual_Cost"], marker="o")
axes[0].set_title("Line: Monthly average actual cost")

# Label axes with units to match engineering intuition.
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Average actual cost")

# Plot grouped averages as a bar plot.
axes[1].bar(by_type["Project_Type"], by_type["Actual_Cost"], color="steelblue")
axes[1].set_title("Bar: Average actual cost by project type")

# Rotate labels for readability and reduce clutter.
axes[1].set_xlabel("Project type")
axes[1].set_ylabel("Average actual cost")

# Improve layout and show the single combined figure.
plt.tight_layout()
plt.show()



### **3.2. Scatter Plot Relationships**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_03_02.jpg?v=1773537776" width="250">



>* Show relationships between two numeric variables.
>* Use clear labels, units, and fair axes.

>* Look for curves, clusters, influential points
>* Investigate outliers; keep grouping visuals simple

>* Use scatter plots to answer one question.
>* Improve readability; add trends, captions, consistent scales.



In [ ]:
#@title Python Code - Scatter Plot Relationships

# Learn scatter plots for civil engineering relationships.
# Use pandas cleaning for missing numeric values.
# Create one clear plot with labels and units.

# Import core libraries for tables and plotting.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Download a small civil engineering style dataset.
!wget -q -O bim.csv "https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv"

# Load the dataset into a pandas dataframe.
df = pd.read_csv("bim.csv")

# Show a few column names for orientation.
print("Columns sample:", list(df.columns)[:8])

# Pick two quantitative columns for a relationship.
x_col, y_col = "Vibration_Level", "Crack_Width"

# Convert chosen columns to numeric safely.
df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
df[y_col] = pd.to_numeric(df[y_col], errors="coerce")

# Keep only rows with both values present.
df_xy = df[[x_col, y_col]].dropna()

# Guard against empty data after cleaning.
if len(df_xy) < 10:
    x_col, y_col = "Planned_Cost", "Actual_Cost"

    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df[y_col] = pd.to_numeric(df[y_col], errors="coerce")

    df_xy = df[[x_col, y_col]].dropna()

# Limit points to keep the plot readable.
df_xy = df_xy.head(300)

# Compute a simple correlation for context.
r = df_xy[x_col].corr(df_xy[y_col])

# Print one short summary line for interpretation.
print("Using:", x_col, "vs", y_col, "corr=", round(r, 3))

# Create a clear scatter plot with transparency.
plt.figure(figsize=(7, 5))
plt.scatter(df_xy[x_col], df_xy[y_col], s=28, alpha=0.55)

# Add labels, title, and a light grid.
plt.xlabel(x_col.replace("_", " "))
plt.ylabel(y_col.replace("_", " "))
plt.title("Scatter plot relationship in a CE dataset")
plt.grid(True, alpha=0.25)

# Set axis limits to show full data range.
plt.xlim(df_xy[x_col].min(), df_xy[x_col].max())
plt.ylim(df_xy[y_col].min(), df_xy[y_col].max())

# Render exactly one plot for the lecture.
plt.show()



### **3.3. Report Ready Styling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_A/image_03_03.jpg?v=1773537803" width="250">



>* Make plots self-contained: labels, units, title, legend.
>* Use readable, uncluttered styles; grayscale-friendly distinctions.

>* Choose scales and grids to show trends honestly.
>* Use reference lines; show gaps, avoid false continuity.

>* Keep plots consistent: colors, units, hierarchy.
>* Ensure legible, accessible visuals for quick decisions.



In [ ]:
#@title Python Code - Report Ready Styling

# Make civil engineering plots readable in reports.
# Apply consistent labels, units, and legend styling.
# Add subtle reference lines for engineering thresholds.

# !pip -q install pandas matplotlib

# Import core plotting and table tools.
import pandas as pd
import matplotlib.pyplot as plt

# Download a small civil engineering style dataset.
!wget -q -O bim.csv "https://raw.githubusercontent.com/mhrafiei/data/main/bim.csv"

# Load data and keep only needed columns.
df = pd.read_csv("bim.csv")
cols = ["Start_Date", "Project_Type", "Vibration_Level"]

# Select columns defensively and drop missing essentials.
df = df.loc[:, cols]
df = df.dropna(subset=["Start_Date", "Project_Type"])

# Convert dates and numeric values for plotting.
df["Start_Date"] = pd.to_datetime(df["Start_Date"], errors="coerce")
df["Vibration_Level"] = pd.to_numeric(df["Vibration_Level"], errors="coerce")

# Remove rows with invalid dates or measurements.
df = df.dropna(subset=["Start_Date", "Vibration_Level"])
df = df.sort_values("Start_Date")

# Keep a small, deterministic time window.
df = df[df["Start_Date"] <= df["Start_Date"].min() + pd.Timedelta(days=120)]
df = df.head(400)

# Summarize by month and project type.
df["Month"] = df["Start_Date"].dt.to_period("M").dt.to_timestamp()
g = df.groupby(["Month", "Project_Type"], as_index=False)["Vibration_Level"].mean()

# Pivot to wide format for clean multi-line plotting.
p = g.pivot(index="Month", columns="Project_Type", values="Vibration_Level")
p = p.dropna(axis=1, how="all")

# Validate we have enough data to plot.
print("Rows used:", int(df.shape[0]), "Monthly points:", int(p.shape[0]))

# Choose a few series to keep the figure readable.
series = list(p.columns)
series = series[:3]

# Create a report-ready static figure.
plt.rcParams.update({"font.size": 11})
fig, ax = plt.subplots(figsize=(9, 4.8))

# Plot each project type with clear line styles.
styles = ["-", "--", ":"]
markers = ["o", "s", "D"]

# Draw lines with markers for grayscale readability.
for i in range(len(series)):
    ax.plot(p.index, p[series[i]], linestyle=styles[i], marker=markers[i], label=series[i])

# Add an engineering threshold reference line.
threshold = float(df["Vibration_Level"].quantile(0.90))
ax.axhline(threshold, color="black", linewidth=1.0, alpha=0.6)

# Annotate the threshold without a long caption.
ax.text(p.index.min(), threshold, "90th percentile reference", va="bottom")

# Add context, units, and clean axis formatting.
ax.set_title("Average Vibration Level by Month and Project Type")
ax.set_xlabel("Start month (calendar date)")
ax.set_ylabel("Vibration level (dataset units)")

# Use subtle gridlines and a concise legend.
ax.grid(True, which="major", linewidth=0.6, alpha=0.25)
ax.legend(title="Project type", ncols=1, frameon=False)

# Improve layout for embedding in reports.
fig.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Pandas & Plots**</font>


In this lecture, you learned to:
- Manipulate civil engineering tables using pandas operations such as filtering, grouping, and joining. 
- Handle missing values and basic data type conversions in preparation for later modeling. 
- Create clear static plots that communicate key trends in civil engineering datasets. 

In the next Lecture (Lecture B), we will go over 'Pandas & Arrays'